In [ ]:
!pip install -q unsloth

In [ ]:
!pip install -q datasets trl accelerate

In [ ]:
from unsloth import FastLanguageModel
import torch
import numpy as np
import re

max_seq_length = 256
dtype = torch.float16
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="mistralai/Mistral-7B-v0.1",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

==((====))==  Unsloth 2026.3.8: Fast Mistral patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",  # faster than HF
)

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "math-ai/StackMathQA",
    "stackmathqa1600k",
)["train"]

dataset = dataset.train_test_split(test_size=0.05, seed=42)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Q', 'A', 'meta'],
        num_rows: 1520000
    })
    test: Dataset({
        features: ['Q', 'A', 'meta'],
        num_rows: 80000
    })
})

In [ ]:
def formatting_func(example):
    text = f"""

### Question:
{example['Q']}

### Answer:
{example['A']}"""

    # tokenize WITHOUT padding
    tokens = tokenizer(
        text,
        truncation=True,
        max_length=256,     # MUST match trainer
        add_special_tokens=True,
    )

    # decode back to truncated text
    truncated_text = tokenizer.decode(tokens["input_ids"])

    return [truncated_text]

In [ ]:
def extract_answer(text: str):
    if text is None:
        return ""

    text = text.strip()

    patterns = [
        r"####\s*(.*)",
        r"Final Answer:\s*(.*)",
        r"Answer:\s*(.*)",
        r"The answer is\s*(.*)",
    ]

    for p in patterns:
        match = re.search(p, text, re.IGNORECASE)
        if match:
            return match.group(1).strip()

    # fallback → last line
    return text.split("\n")[-1].strip()

In [ ]:
def normalize_answer(s):
    return (
        s.lower()
        .replace(" ", "")
        .replace(",", "")
        .replace("$", "")
    )


def exact_match(pred, gold):
    return normalize_answer(pred) == normalize_answer(gold)

In [ ]:
def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # logits -> token ids
    pred_ids = np.argmax(preds, axis=-1)

    # replace ignored tokens
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    pred_texts = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_texts = tokenizer.batch_decode(labels, skip_special_tokens=True)

    correct = 0
    total = len(pred_texts)

    for pred, gold in zip(pred_texts, label_texts):
        pred_ans = extract_answer(pred)
        gold_ans = extract_answer(gold)

        if exact_match(pred_ans, gold_ans):
            correct += 1

    return {
        "exact_match": correct / total
    }

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    formatting_func=formatting_func,
    compute_metrics=compute_metrics,
    max_seq_length=256,
    packing=True,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=50,
        num_train_epochs=3,
        learning_rate=2e-5,
        fp16=True,
        logging_steps=20,
        optim="adamw_8bit",
        weight_decay=0.01,
        eval_strategy="steps",
        eval_steps=100,
        lr_scheduler_type="linear",
        output_dir="outputs",
        report_to="none",
    ),
)

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1520000 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/80000 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,524 | Num Epochs = 3 | Total steps = 573
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,283,675,136 (0.58% trained)


Step,Training Loss,Validation Loss,Exact Match
100,1.499548,1.518056,0.000000
200,1.481912,1.494855,0.000000
300,1.396146,1.479391,0.000000
400,1.394285,1.470701,0.000000
500,1.430117,1.468518,0.000000


Unsloth: Not an error, but MistralForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transfo

In [ ]:
model.save_pretrained("model_name")
tokenizer.save_pretrained("model_name")

In [ ]:
prompt = """### Question:
Solve lim(x->0)sin(3x)/x

### Answer:
"""

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=200
)

print(tokenizer.decode(outputs[0]))